# 02 — Feature Engineering

This notebook **explores and visualises** the features defined in `src/features.py`.
The actual computation logic lives there — this notebook just calls it and shows what the features look like.

**What you'll see here:**
- What each feature looks like plotted over time
- Correlation between features (checking for redundancy)
- Distribution of the prediction target
- A note on lookahead bias and how we avoid it

In [ ]:
import sys
sys.path.append('..')  # so Python can find the src/ folder

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import from src/ — this is the key pattern
from src.data_pipeline import load_ticker
from src.features import build_features, get_X_y, FEATURE_COLS

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## Load raw data and build features

We use AAPL as a representative stock for exploration.
Note that we call `build_features()` — a single function from `src/features.py` that
runs the entire pipeline. We don't reimplement any logic here.

In [ ]:
raw = load_ticker("AAPL")
print(f"Raw data: {raw.shape[0]} rows from {raw.index[0].date()} to {raw.index[-1].date()}")
raw.head()

In [ ]:
features = build_features(raw)
print(f"After feature engineering: {features.shape[0]} rows, {features.shape[1]} columns")
print(f"Rows dropped during warmup: {raw.shape[0] - features.shape[0]} (expected — rolling windows need history)")
features.head()

## Visualise individual features

Before trusting a feature, it's worth plotting it. We want to check:
- Does it look sensible? (no obvious bugs)
- Is it stationary-ish, or does it drift in a way that could leak regime info?
- Does it have the dynamic range we'd expect?

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1: Price with moving averages
features['Close'].plot(ax=axes[0], color='steelblue', linewidth=1, label='Close')
features['ma_10'].plot(ax=axes[0], color='orange', linewidth=1.2, label='MA 10')
features['ma_50'].plot(ax=axes[0], color='red', linewidth=1.2, label='MA 50')
axes[0].set_title('AAPL Close Price with Moving Averages')
axes[0].legend()

# Panel 2: RSI
features['rsi_14'].plot(ax=axes[1], color='purple', linewidth=1)
axes[1].axhline(70, color='red', linestyle='--', linewidth=0.8, label='Overbought (70)')
axes[1].axhline(30, color='green', linestyle='--', linewidth=0.8, label='Oversold (30)')
axes[1].set_title('RSI (14-day)')
axes[1].set_ylim(0, 100)
axes[1].legend()

# Panel 3: Rolling volatility
features['vol_21d'].plot(ax=axes[2], color='darkgreen', linewidth=1, label='21d Vol')
features['vol_10d'].plot(ax=axes[2], color='lightgreen', linewidth=1, label='10d Vol')
axes[2].set_title('Rolling Volatility (annualised)')
axes[2].legend()

plt.tight_layout()
plt.savefig('../notebooks/figures/feature_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature correlation matrix

High correlation between features isn't necessarily fatal, but it's worth knowing about.
Highly correlated features add redundancy without adding information — and can make
model weights harder to interpret.

In [ ]:
X, y = get_X_y(features)

corr = X.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    square=True, linewidths=0.5,
    ax=ax
)
ax.set_title('Feature Correlation Matrix — AAPL')
plt.tight_layout()
plt.savefig('../notebooks/figures/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Flag any pairs with correlation > 0.8
high_corr = [
    (c1, c2, corr.loc[c1, c2])
    for i, c1 in enumerate(FEATURE_COLS)
    for c2 in FEATURE_COLS[i+1:]
    if abs(corr.loc[c1, c2]) > 0.8
]
if high_corr:
    print("High-correlation pairs (|r| > 0.8):")
    for c1, c2, r in high_corr:
        print(f"  {c1} <-> {c2}: {r:.3f}")
else:
    print("No highly correlated feature pairs found.")

## Distribution of the prediction target

Understanding the target distribution matters before choosing a loss function.
Stock returns are roughly normal but with fat tails — the extreme events are
more common than a Gaussian would suggest.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
y.hist(bins=80, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Distribution of 5-Day Forward Returns (AAPL)')
axes[0].set_xlabel('Return')

# Over time — useful to spot regime changes
y.rolling(63).std().plot(ax=axes[1], color='darkorange')
axes[1].set_title('Rolling 63-Day Std of Target (regime check)')
axes[1].set_ylabel('Std of 5d Return')

plt.tight_layout()
plt.show()

print(f"Target stats:")
print(y.describe().round(4))
print(f"Skewness : {y.skew():.3f}")
print(f"Kurtosis : {y.kurtosis():.3f}  (>3 = fat tails)")

## Lookahead bias check

Lookahead bias is when a feature accidentally uses future information.
It's the most common mistake in financial ML and produces falsely optimistic results.

Our guard: every feature at time `t` should only use data from `t` and earlier.
The `target_5d` column is the *only* column that looks forward — and it's never
included in `FEATURE_COLS`.

Verify that no feature column uses `.shift(-n)` by inspecting `src/features.py` directly.
This is something worth explicitly calling out in your README.

In [ ]:
# Sanity check: confirm target is NOT in FEATURE_COLS
from src.features import TARGET_COL

assert TARGET_COL not in FEATURE_COLS, "BUG: target column is in feature list!"
print(f"✓ '{TARGET_COL}' is not in FEATURE_COLS — lookahead bias check passed.")
print(f"\nFeature columns ({len(FEATURE_COLS)}):")
for col in FEATURE_COLS:
    print(f"  {col}")